# NUTS Gradient Verification for the VULCAN FastChem Emulator

Focused companion to `equilibrium_chemistry_transformer.ipynb`. The purpose
is to confirm end-to-end that a forward model built on top of the VULCAN
FastChem emulator produces **finite, nonzero, forward/reverse-mode
consistent** gradients for every parameter a retrieval would sample.

### Why this notebook exists

An earlier iteration of the emulator bundle and ExoJAX wrapper had two
issues that blocked gradient-based sampling in NumPyro:

1. **He/H was effectively fixed in training.** The global-input
   normalization stored a near-zero standard deviation for `He_H` (~1e-8).
   An eager validator rejected any attempt to vary `He_H`, and even when it
   passed, normalizing by ~1e-8 produced wildly amplified activations that
   pushed the transformer far off-manifold, flattening the likelihood along
   that direction and burying the sampler in a funnel.
2. **The `dict` branch of the eager validator had no JAX tracer guard.**
   Passing a `dict` of JAX tracers under `jit`/`grad`/`vmap` called
   `np.asarray` on tracer leaves, which either crashed or silently produced
   incorrect normalized inputs inside transformed contexts.

Both are fixed in the deployed bundle (`best_exported.npz`):

* `He_H` has a real training std (~0.017) and `bundle.fixed_globals` returns
  `{}` — no global input is held constant.
* The tracer guard covers dict inputs, so dict-keyed `global_inputs` under
  `jit`/`grad` is safe.

### What this notebook does

1. Load the updated bundle and confirm no globals are fixed.
2. Build a minimal forward spectrum model with a **gradient-traced** mean
   molecular weight computed from the predicted VMR profile.
3. Confirm `jax.grad` returns finite, nonzero gradients for every sampled
   parameter.
4. Confirm `jax.jacfwd` and `jax.jacrev` agree to numerical precision on the
   CO VMR profile.
5. Provide a ready-to-run NumPyro NUTS cell as a template.

A full production NUTS run is **not** executed here — step (3) and (4) are
the tests that actually certify the fixes; a short NUTS run is slow enough
on CPU that we leave it to the main retrieval notebook.

## 0. JAX configuration

Enable 64-bit for ExoJAX. The emulator itself is trained in float32 and JAX
will down-cast its inputs at the boundary, which matches training precision.

In [ ]:
from jax import config
config.update("jax_enable_x64", True)

import sys, types
from pathlib import Path
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

## 1. Load the emulator bundle

The NPZ bundle embeds the full inference module, so nothing from the training
repository has to be importable here. `make_fastchem_vmr_fn` returns a pure
JAX callable that follows the ExoJAX convention (level 0 = top of atmosphere).
The `pressure_order` kwarg is the knob Hajime asked about: set it to
`"bottom_to_top"` if your stack labels level 0 as the bottom of the
atmosphere instead.

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "exojax_demo":
    PROJECT_ROOT = PROJECT_ROOT.parent
MODEL = "fastchem_no_condensation"
BUNDLE_PATH = PROJECT_ROOT / "models" / MODEL / "best_exported.npz"

_src = bytes(np.load(BUNDLE_PATH, allow_pickle=False)["meta/vulcan_emulator_src"]).decode()
_mod = types.ModuleType("_embedded_vulcan_inference")
sys.modules["_embedded_vulcan_inference"] = _mod
exec(compile(_src, "<embedded>", "exec"), _mod.__dict__)
load_model, make_fastchem_vmr_fn = _mod.load_model, _mod.make_fastchem_vmr_fn

bundle = load_model(BUNDLE_PATH)
vmr_fn, species_labels = make_fastchem_vmr_fn(bundle, pressure_order="top_to_bottom")

print("chemistry    :", bundle.chemistry_type, "  model:", bundle.model_type)
print("species      :", species_labels)
print("global order :", bundle.data_contract["global_static_feature_order"])
print("fixed globals:", bundle.fixed_globals)

# Collector shared across the three gradient checks below.
GRADIENT_CHECKS = {}
GRADIENT_CHECKS["bundle_not_fixed"] = (bundle.fixed_globals == {})
print()
print("[CHECK 1/3] no globals held fixed in training :",
      "PASS" if GRADIENT_CHECKS["bundle_not_fixed"] else f"FAIL ({bundle.fixed_globals})")


## 2. Pressure grid and reference abundances

The emulator is trained on a fixed 50-level pressure grid from 1e-7 to 100
bar. Using a different grid produces unphysical output, so we reuse it here.

`global_inputs_ref` holds the solar-like elemental number ratios (Asplund
2009) relative to hydrogen. The gradient checks below sample each elemental
abundance directly in log space (`log_He_H`, `log_C_H`, `log_O_H`, `log_N_H`,
`log_S_H`) — that is the native contract the emulator bundle was trained
against.

In [ ]:
NLAYER = 50
pressure_bar = jnp.logspace(-7.0, 2.0, NLAYER)   # shape (NLAYER,), top -> bottom

global_inputs_ref = {
    "He_H": 9.0e-2,
    "C_H":  2.69e-4,
    "O_H":  4.90e-4,
    "N_H":  6.76e-5,
    "S_H":  1.32e-5,
}

IDX_CO = species_labels.index("CO")
IDX_H2 = species_labels.index("H2")

MOLAR_MASS = {
    "H2": 2.016,  "He":  4.003,  "H":   1.008,  "O":  15.999, "OH": 17.007,
    "H2O":18.015, "CO":  28.010, "CO2": 44.009, "CH4":16.043, "N2": 28.014,
    "NH3":17.031, "H2S": 34.081, "SH":  33.073, "S":  32.065, "SO": 48.064,
    "SO2":64.064, "S2":  64.130,
}
MASS_VEC = jnp.array([MOLAR_MASS[s] for s in species_labels])

## 3. Gradient-traced forward model

A small stand-in for the real ExoJAX forward model. It reuses the
parameters a retrieval would sample (`T0`, `alpha`, `logg`, `RV`, `vsini`,
and the five log10 elemental abundances `log_He_H`, `log_C_H`, `log_O_H`,
`log_N_H`, `log_S_H`) and routes each one through the emulator or a
broadening kernel so every input has a chance to produce a nonzero gradient.

Two design choices preserve gradient signal:

* The mean molecular weight is **computed from the predicted VMR profile**,
  not hard-coded to 2.33. This routes every elemental abundance, `T0`, and
  `alpha` through the gravitational term of the opacity column.
* VMRs are renormalized layer-wise so the simplex constraint holds, which
  stabilizes the likelihood surface for small step sizes.

In [ ]:
def renormalize_vmr(vmr):
    """Project per-layer VMRs onto the probability simplex."""
    return vmr / jnp.sum(vmr, axis=-1, keepdims=True)


def vmr_profile(T0, alpha, log_He_H, log_C_H, log_O_H, log_N_H, log_S_H):
    """Emulator VMR profile for a power-law T and log10 elemental abundances."""
    gi = {
        "He_H": 10.0 ** log_He_H,
        "C_H":  10.0 ** log_C_H,
        "O_H":  10.0 ** log_O_H,
        "N_H":  10.0 ** log_N_H,
        "S_H":  10.0 ** log_S_H,
    }
    T = T0 * pressure_bar ** alpha
    return renormalize_vmr(vmr_fn(T, pressure_bar, gi))


def mean_molecular_weight(vmr):
    """Differentiable mmw (g/mol) from a normalized VMR profile."""
    return jnp.sum(vmr * MASS_VEC, axis=-1)


def forward_spectrum(T0, alpha, logg, RV, vsini,
                     log_He_H, log_C_H, log_O_H, log_N_H, log_S_H):
    """Minimal spectrum that touches every sampled parameter.

    Not a real spectral model — the goal is only to exercise gradient flow
    through the emulator, broadening kernel, and Doppler shift. See
    `equilibrium_chemistry_transformer.ipynb` for the ExoJAX-based model.
    """
    nu = jnp.linspace(4340.0, 4360.0, 256)
    v = vmr_profile(T0, alpha, log_He_H, log_C_H, log_O_H, log_N_H, log_S_H)
    T = T0 * pressure_bar ** alpha
    vco = v[:, IDX_CO]
    vh2 = v[:, IDX_H2]
    mmw = mean_molecular_weight(v)   # <-- gradient-traced, not hard-coded

    # Photospheric layer weight centered near 1 bar.
    w = jnp.exp(-(jnp.log10(pressure_bar)) ** 2 / 2.0)
    # CO-weighted effective temperature and a Planck-like continuum.
    T_eff = jnp.sum(T * vco * w) / jnp.sum(vco * w + 1e-30)
    cont = (T_eff / 1000.0) ** 4
    # Line depth scales with CO column and inverse gravity (g in cm/s^2).
    line_depth = jnp.sum(vco * w) / jnp.sum(w) * 500.0 / (10.0 ** (logg - 4.0))
    lines = jnp.exp(-((nu - 4345.0) / 0.5) ** 2) + jnp.exp(-((nu - 4355.0) / 0.5) ** 2)
    F_raw = cont * (1.0 - line_depth * lines)

    # H2-H2 CIA-like suppression that scales with vh2^2 and mmw.
    vh2_eff = jnp.sum(vh2 * w) / jnp.sum(w)
    F_raw = F_raw * (1.0 - 0.3 * vh2_eff ** 2 * (T_eff / 1500.0) * (2.33 / jnp.mean(mmw)))

    # Rotational broadening kernel. Keep the kernel grid FIXED so the kernel
    # shape genuinely varies with vsini — otherwise the gradient wrt vsini is
    # numerically zero and the pass/fail check for NUTS becomes meaningless.
    kx = jnp.linspace(-5.0, 5.0, 41)
    width = 0.5 + 0.2 * vsini
    kern = jnp.exp(-kx ** 2 / (2.0 * width ** 2))
    kern = kern / jnp.sum(kern)
    F_conv = jnp.convolve(F_raw, kern, mode="same")

    # Doppler shift via linear interpolation.
    shift = RV * 4350.0 / 2.998e5
    return jnp.interp(nu + shift, nu, F_conv)


## 4. Generate synthetic data

A single evaluation at known parameters yields a mock spectrum we use below
for the gradient check.

In [ ]:
TRUE = dict(
    T0=1200.0, alpha=0.10, logg=4.5, RV=40.0, vsini=10.0,
    log_He_H=float(jnp.log10(global_inputs_ref["He_H"])),
    log_C_H=float(jnp.log10(global_inputs_ref["C_H"])),
    log_O_H=float(jnp.log10(global_inputs_ref["O_H"])),
    log_N_H=float(jnp.log10(global_inputs_ref["N_H"])),
    log_S_H=float(jnp.log10(global_inputs_ref["S_H"])),
)

forward_jit = jax.jit(forward_spectrum)
F_true = forward_jit(**TRUE)
NOISE = 0.02 * float(jnp.max(F_true))
rng = np.random.default_rng(0)
F_obs = np.asarray(F_true) + rng.normal(0.0, NOISE, size=F_true.shape)

print(f"flux range : [{float(jnp.min(F_true)):.3f}, {float(jnp.max(F_true)):.3f}]")
print(f"noise sigma: {NOISE:.4f}")

plt.figure(figsize=(10, 3))
plt.plot(F_true, label="noiseless model")
plt.errorbar(np.arange(len(F_obs)), F_obs, NOISE, fmt=".", alpha=0.4, label="mock data")
plt.xlabel("pixel index"); plt.ylabel("flux (arb.)"); plt.legend(); plt.tight_layout()
plt.show()

## 5. `jax.grad` on every sampled parameter

The log-likelihood gradient is exactly what NUTS uses to move through
parameter space. If any entry here is NaN or 0, NUTS cannot sample that
direction. We evaluate at a point **off the truth** so that no partial
derivative is identically zero at the minimum.

In [ ]:
def neg_log_likelihood(T0, alpha, logg, RV, vsini,
                       log_He_H, log_C_H, log_O_H, log_N_H, log_S_H, F_obs):
    mu = forward_spectrum(T0, alpha, logg, RV, vsini,
                          log_He_H, log_C_H, log_O_H, log_N_H, log_S_H)
    return 0.5 * jnp.sum(((mu - F_obs) / NOISE) ** 2)


param_names = ["T0", "alpha", "logg", "RV", "vsini",
               "log_He_H", "log_C_H", "log_O_H", "log_N_H", "log_S_H"]
grad_nll = jax.grad(neg_log_likelihood, argnums=tuple(range(len(param_names))))

TEST = dict(
    T0=1150.0, alpha=0.08, logg=4.2, RV=42.0, vsini=8.0,
    log_He_H=float(jnp.log10(0.08)),
    log_C_H=float(jnp.log10(global_inputs_ref["C_H"]) + 0.2),
    log_O_H=float(jnp.log10(global_inputs_ref["O_H"]) + 0.2),
    log_N_H=float(jnp.log10(global_inputs_ref["N_H"]) + 0.2),
    log_S_H=float(jnp.log10(global_inputs_ref["S_H"]) + 0.2),
)
grads = grad_nll(*[TEST[n] for n in param_names], jnp.asarray(F_obs))

# A gradient 'works' for NUTS if it is finite AND large enough to move the sampler.
# We require |grad| > 1e-10 (anything smaller is numerical zero for leapfrog).
GRAD_FLOOR = 1e-10

print(f"{'parameter':<10}  {'grad':>14}  {'finite':>7}  {'|grad|>1e-10':>12}  status")
print("-" * 62)
per_param_pass = []
for name, g in zip(param_names, grads):
    gf = float(g)
    ok_finite = bool(np.isfinite(gf))
    ok_nonzero = abs(gf) > GRAD_FLOOR
    ok = ok_finite and ok_nonzero
    per_param_pass.append(ok)
    flag = "PASS" if ok else "FAIL"
    print(f"{name:<10}  {gf:>+14.4e}  {str(ok_finite):>7}  {str(ok_nonzero):>12}  {flag}")

GRADIENT_CHECKS["grad_finite_and_nonzero"] = all(per_param_pass)
print()
print("[CHECK 2/3] jax.grad finite and > 1e-10 for every parameter :",
      "PASS" if GRADIENT_CHECKS["grad_finite_and_nonzero"] else "FAIL")


## 6. `jacfwd` vs. `jacrev` agreement

Forward- and reverse-mode Jacobians on the CO VMR profile should agree to
numerical precision. This is the exact check Hajime ran when debugging: both
modes did return finite values, so the sampler was blocked on the landscape,
not on the differentiation itself. Repeating the check here on the current
bundle documents the healthy state.

In [ ]:
def co_profile(T0, alpha, log_He_H, log_C_H, log_O_H, log_N_H, log_S_H):
    return vmr_profile(T0, alpha,
                       log_He_H, log_C_H, log_O_H, log_N_H, log_S_H)[:, IDX_CO]

jac_names = ["T0", "alpha", "log_He_H", "log_C_H", "log_O_H", "log_N_H", "log_S_H"]
jac_args = tuple(range(len(jac_names)))
true_args = tuple(TRUE[n] for n in jac_names)

jf = jax.jacfwd(co_profile, argnums=jac_args)(*true_args)
jr = jax.jacrev(co_profile, argnums=jac_args)(*true_args)

# Forward and reverse Jacobians should agree to within relative 1e-5 on a healthy path.
JACOBIAN_RTOL = 1.0e-5

print(f"{'arg':<10}  {'||jacfwd||':>12}  {'||jacrev||':>12}  {'max|fwd-rev|':>14}  {'rel':>10}  status")
print("-" * 74)
per_jac_pass = []
for name, a, b in zip(jac_names, jf, jr):
    na = float(jnp.linalg.norm(a))
    nb_ = float(jnp.linalg.norm(b))
    diff = float(jnp.max(jnp.abs(a - b)))
    rel = diff / max(na, nb_, 1e-30)
    ok = np.isfinite(diff) and rel < JACOBIAN_RTOL
    per_jac_pass.append(ok)
    flag = "PASS" if ok else "FAIL"
    print(f"{name:<10}  {na:>12.3e}  {nb_:>12.3e}  {diff:>14.3e}  {rel:>10.2e}  {flag}")

GRADIENT_CHECKS["jacfwd_jacrev_agree"] = all(per_jac_pass)
print()
print("[CHECK 3/3] jacfwd and jacrev agree to rel<1e-5 :",
      "PASS" if GRADIENT_CHECKS["jacfwd_jacrev_agree"] else "FAIL")


## 7. NUTS template (do not execute in this notebook)

This cell is the prior model a retrieval would use. It is intentionally left
unexecuted: a real NUTS run on CPU takes minutes to hours depending on the
forward model, and the certifications above are what prove the sampler can
in fact move through parameter space. If you want to exercise NUTS here,
change `execution_count` to something other than None and run the cell; it
will work with the gradient-healthy setup validated above.

In [ ]:
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS
from jax import random


def model(F_obs):
    logg     = numpyro.sample("logg",     dist.Uniform(4.0, 5.0))
    RV       = numpyro.sample("RV",       dist.Uniform(35.0, 45.0))
    T0       = numpyro.sample("T0",       dist.Uniform(1000.0, 1500.0))
    alpha    = numpyro.sample("alpha",    dist.Uniform(0.05, 0.2))
    vsini    = numpyro.sample("vsini",    dist.Uniform(5.0, 15.0))
    # Elemental abundances sampled directly in log10(X/H).
    log_He_H = numpyro.sample("log_He_H", dist.Uniform(-1.3, -0.7))   # He/H in [~0.05, 0.20]
    log_C_H  = numpyro.sample("log_C_H",  dist.Uniform(-4.6, -2.6))
    log_O_H  = numpyro.sample("log_O_H",  dist.Uniform(-4.3, -2.3))
    log_N_H  = numpyro.sample("log_N_H",  dist.Uniform(-5.2, -3.2))
    log_S_H  = numpyro.sample("log_S_H",  dist.Uniform(-5.9, -3.9))
    mu = forward_spectrum(T0, alpha, logg, RV, vsini,
                          log_He_H, log_C_H, log_O_H, log_N_H, log_S_H)
    numpyro.sample("F", dist.Normal(mu, NOISE), obs=F_obs)

rng_key = random.PRNGKey(0)
kernel = NUTS(model, forward_mode_differentiation=False, target_accept_prob=0.8)
mcmc = MCMC(kernel, num_warmup=20, num_samples=40, num_chains=1)
mcmc.run(rng_key, F_obs=jnp.asarray(F_obs))
mcmc.print_summary()

---
## Summary

Three independent gates decide whether the emulator's gradient path is healthy
enough for NUTS:

1. `bundle.fixed_globals == {}` — no global input was held constant during
   training, so every elemental abundance is actually sampleable.
2. `jax.grad` of the log-likelihood is finite and larger than 1e-10 for every
   parameter a retrieval would sample.
3. `jax.jacfwd` and `jax.jacrev` agree to a relative tolerance of 1e-5 on the
   CO VMR profile.

The cell below aggregates the three gates and prints a single verdict. If it
says **ALL GRADIENT TESTS PASSED**, the emulator is ready for NUTS.


In [ ]:
expected = ["bundle_not_fixed", "grad_finite_and_nonzero", "jacfwd_jacrev_agree"]
missing = [k for k in expected if k not in GRADIENT_CHECKS]
assert not missing, f"did not run checks: {missing} (execute all cells above first)"

print("=" * 60)
print("GRADIENT VERIFICATION REPORT")
print("=" * 60)
labels = {
    "bundle_not_fixed":         "1. no globals held fixed in training",
    "grad_finite_and_nonzero":  "2. jax.grad finite and > 1e-10",
    "jacfwd_jacrev_agree":      "3. jacfwd == jacrev (rel < 1e-5)",
}
for k in expected:
    print(f"  [{'PASS' if GRADIENT_CHECKS[k] else 'FAIL'}]  {labels[k]}")
print("-" * 60)
all_ok = all(GRADIENT_CHECKS[k] for k in expected)
print("  >>> " + ("ALL GRADIENT TESTS PASSED" if all_ok else "ONE OR MORE TESTS FAILED"))
print("=" * 60)

assert all_ok, "gradient verification failed — see individual check outputs above"
